# Predictor de Dostoyevski — Generador de texto con RNNs

**Inteligencia Artificial: Aprendizaje Automático — Laboratorio de NLP con RNNs**

Universidad Católica Andrés Bello · Escuela de Informática · Semestre Marzo–Julio 2026

**Estudiantes:** William Serrano (C.I. 28.148.935) y Eduardo Rojas (C.I. 31.127.326)

---

## Objetivo

Implementar un generador de texto basado en una **RNN a nivel de caracteres** (*char-RNN*),
entrenado con la novela *Crimen y Castigo* de Fiódor Dostoyevski, capaz de producir texto
parecido al del libro.

El entregable principal es la función:

```python
generarTexto(secuencia)  # recibe un string de 25 caracteres -> retorna 50 caracteres generados
```


## 1. Configuración de la plataforma

Se usa Keras 2 mediante la variable de entorno `TF_USE_LEGACY_KERAS` y el paquete `tf_keras`. Esto hace que `tf.keras` apunte a Keras 2,
necesario para que las RNN stateful y `TextVectorization` funcionen como en el ejemplo.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # se usa Keras 2
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # reduce mensajes de log de TF

import tf_keras
import tensorflow as tf
import numpy as np

from packaging import version
assert version.parse(tf.__version__) >= version.parse("2.8.0")

tf.random.set_seed(42)     # reproducibilidad
np.random.seed(42)

print("TensorFlow:", tf.__version__)
print("tf.keras  :", tf.keras.__name__, "(Keras 2)")


TensorFlow: 2.21.0
tf.keras  : tf_keras.api._v2.keras (Keras 2)


## 2. Carga del corpus (*Crimen y Castigo*)

In [ ]:
# from google.colab import files; files.upload()

ruta = "./crimenYcastigo.txt"

with open(ruta, encoding="utf-8") as f:
    texto_completo = f.read().replace("\ufeff", "")  # se quita el BOM inicial

print("Archivo cargado desde:", ruta)
print("Cantidad total de caracteres:", len(texto_completo))

Archivo cargado desde: ./crimenYcastigo.txt
Cantidad total de caracteres: 1052268


Caracteres distintos presentes en el corpus (incluye acentos, ñ, signos `¿ ¡` y comillas
`« »` propios del español):

In [3]:
distintos = sorted(set(texto_completo.lower()))
print("Número de caracteres distintos (en minúscula):", len(distintos))
print("".join(distintos))

Número de caracteres distintos (en minúscula): 66

 !()*,-.0123456789:;?[]_abcdefghijklmnopqrstuvwxyz¡«»¿áäéíïñóöúûü


## 3. Vectorización a nivel de carácter

`TextVectorization(split="character", standardize="lower")` divide el texto en caracteres, los pasa a **minúscula** y asigna a cada uno un entero. Igual que en clase:

- El índice `0` se reserva para *padding* y el `1` para caracteres desconocidos (`[UNK]`).
- Se restan `2` para que los caracteres reales empiecen en `0`; así la capa de salida tiene
  exactamente `n_tokens` neuronas.


In [4]:
text_vec_layer = tf.keras.layers.TextVectorization(
    split="character", standardize="lower"
)
text_vec_layer.adapt([texto_completo])

encoded = text_vec_layer([texto_completo])[0]
encoded -= 2                                      
n_tokens = text_vec_layer.vocabulary_size() - 2   
dataset_size = len(encoded)                       

print(f"Número de caracteres distintos (n_tokens): {n_tokens}")
print(f"Cantidad total de caracteres (dataset_size): {dataset_size}")



Número de caracteres distintos (n_tokens): 66
Cantidad total de caracteres (dataset_size): 1052268


## 4. Construcción del conjunto de datos (ventanas)

In [5]:
def to_dataset(sequence, length, shuffle=False, seed=None, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(sequence)
    ds = ds.window(length + 1, shift=1, drop_remainder=True)
    ds = ds.flat_map(lambda window_ds: window_ds.batch(length + 1))
    if shuffle:
        ds = ds.shuffle(100_000, seed=seed)
    ds = ds.batch(batch_size)
    return ds.map(lambda window: (window[:, :-1], window[:, 1:])).prefetch(1)

Se usan ventanas de longitud `length = 100` para el entrenamiento (más contexto = mejor
aprendizaje). En la generación, la función `generarTexto` recibirá semillas de 25 caracteres;
la RNN maneja secuencias de longitud variable sin problema.

In [6]:
length = 100
n = len(encoded)
i_train = int(n * 0.90)
i_valid = int(n * 0.95)

tf.random.set_seed(42)
train_set = to_dataset(encoded[:i_train],          length=length, shuffle=True, seed=42)
valid_set = to_dataset(encoded[i_train:i_valid],   length=length)
test_set  = to_dataset(encoded[i_valid:],          length=length)

print("Tamaños aproximados (caracteres) -> "
      f"train: {i_train:,} | valid: {i_valid - i_train:,} | test: {n - i_valid:,}")

Tamaños aproximados (caracteres) -> train: 947,041 | valid: 52,613 | test: 52,614


## 5. Modelo Char-RNN y entrenamiento

In [ ]:
EPOCHS = 10 
RUTA_MODELO = "modelo_dostoyevski.keras"

if os.path.exists(RUTA_MODELO):
    print(f"Cargando modelo existente desde '{RUTA_MODELO}' (no se reentrena)...")
    model = tf.keras.models.load_model(RUTA_MODELO)
else:
    tf.random.set_seed(42)
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=n_tokens, output_dim=16),
        tf.keras.layers.GRU(128, return_sequences=True),
        tf.keras.layers.Dense(n_tokens, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer="nadam", metrics=["accuracy"])

    model_ckpt = tf.keras.callbacks.ModelCheckpoint(
        RUTA_MODELO, monitor="val_accuracy", save_best_only=True
    )
    history = model.fit(train_set, validation_data=valid_set,
                        epochs=EPOCHS, callbacks=[model_ckpt])

model.summary()


Epoch 1/10



## 6. Modelo final (incluye la vectorización)

El modelo entrenado espera enteros. Se envuelve junto con la capa `TextVectorization` (y una
capa `Lambda` que resta 2) para poder pasarle texto directamente.

In [ ]:
modelo_dostoyevski = tf.keras.Sequential([
    text_vec_layer,
    tf.keras.layers.Lambda(lambda X: X - 2),
    model,
])

Prueba rápida: predecir el siguiente carácter más probable para una frase.

In [ ]:
entrada = tf.constant(["raskolnikoff salió de su "])
y_proba = modelo_dostoyevski.predict(entrada, verbose=0)[0, -1]
y_pred = tf.argmax(y_proba)
print("Carácter más probable:", repr(text_vec_layer.get_vocabulary()[int(y_pred) + 2]))

## 7. Muestreo con temperatura

In [ ]:
def next_char(text, temperature=1.0):
    y_proba = modelo_dostoyevski.predict(tf.constant([text]), verbose=0)[0, -1:]
    rescaled_logits = tf.math.log(y_proba) / temperature
    char_id = tf.random.categorical(rescaled_logits, num_samples=1)[0, 0]
    return text_vec_layer.get_vocabulary()[int(char_id) + 2]

## 8. Función requerida: `generarTexto`

Esta es la función pedida en el enunciado. Recibe un **string de 25 caracteres** y devuelve
**50 caracteres generados** (la continuación generada, no la semilla). Internamente va
prediciendo carácter por carácter y reusando el texto acumulado como contexto.

In [ ]:
def generarTexto(secuencia, temperature=0.5):
    contexto = secuencia
    generado = ""
    for _ in range(50):
        c = next_char(contexto, temperature)
        contexto += c
        generado += c
    return generado

utilidad para tomar fragmentos reales de 25 caracteres del libro y usarlos como
semillas en los ejemplos.

In [ ]:
def semilla_del_libro(longitud=25, seed=None):
    rng = np.random.default_rng(seed)
    i = int(rng.integers(0, len(texto_completo) - longitud))
    return texto_completo[i:i + longitud]

## 9. Ejemplos del generador

In [ ]:
ejemplos = [
    "una tarde muy calurosa de",
    "raskolnikoff se detuvo en",
    "no quiero esto decir que ",
    "la pobreza le aniquilaba ",
]

for semilla in ejemplos:
    semilla = semilla[:25]
    generado = generarTexto(semilla, temperature=0.5)
    print("SEMILLA (25) :", repr(semilla))
    print("GENERADO (50):", repr(generado))
    print("-" * 70)

Mismo punto de partida, distintas temperaturas, para ver el efecto en la creatividad:

In [ ]:
semilla = "una tarde muy calurosa de" 
for t in [0.2, 0.5, 1.0]:
    print(f"temperatura={t}:")
    print("  ", repr(generarTexto(semilla, temperature=t)))
    print()

Comprobación de que el texto generado no copia la continuación literal del libro:

In [ ]:
import textwrap
semilla = semilla_del_libro(25, seed=7)
pos = texto_completo.find(semilla)
real = texto_completo[pos + 25: pos + 75].replace("\n", " ")
gen  = generarTexto(semilla, temperature=0.5)

print("SEMILLA           :", repr(semilla))
print("CONTINUACIÓN REAL :", repr(real))
print("TEXTO GENERADO    :", repr(gen))
print("¿Son idénticos?   :", real.lower() == gen.lower())